# 36 PGP -> 21 IGP ML pipeline

## Imports and definitions

In [ ]:
import os
from collections import defaultdict

import numpy as np
import optuna  # Bayesian hyperparam optimization.
import pandas as pd
import xgboost as xgb  # More robust (in many ways) than native scikit's.
from optuna.samplers import TPESampler
from pytorch_tabnet.tab_model import TabNetRegressor
from scipy.stats import pearsonr, randint, bootstrap
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import (
    ElasticNet,
    HuberRegressor,
    LinearRegression,
    TheilSenRegressor,
)
from sklearn.metrics import make_scorer, root_mean_squared_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from tqdm.notebook import tqdm_notebook  # progress bar.


c:\Users\Denis\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.chdir("D:\\Projects\\GPD\\tabnet")

We need to define some of metrics here.

- `rsq` --- pearson's $r$-squared ($r^2$). Default $R^2$ in scikit is not the same.
- `rmse` --- root mean squared error is in scikit. We just rename it for convinience.
- `objective` --- we transform these two metrics above into a single scalar, giving more weight to `rsq`.

In [ ]:
def _rsq(y, ypred):
    return pearsonr(y, ypred).statistic ** 2


def _objective(y, ypred):
    return -_rsq(y, ypred) + 0.1 * root_mean_squared_error(y, ypred)


rsq = make_scorer(_rsq, greater_is_better=True)
rmse = make_scorer(root_mean_squared_error, greater_is_better=False)
objective = make_scorer(_objective, greater_is_better=False)
mse = make_scorer(mean_squared_error, greater_is_better=False)

### Data import

In [ ]:
predictor_cols = [f"PGP{i}" for i in range(1, 118)]
target_cols = [f"GP{i}" for i in range(1, 22)]
twins = pd.read_csv("twins117.csv")
twins_test, twins_cv = twins.loc[twins.Fold == 0], twins.loc[twins.Fold != 0]

xs_cv, ys_cv = twins_cv[predictor_cols].to_numpy(), twins_cv[target_cols].to_numpy()
xs_test, ys_test = twins_test[predictor_cols].to_numpy(), twins_test[target_cols].to_numpy()

splits = [(np.where(twins_cv.Fold != i)[0], np.where(twins_cv.Fold == i)[0]) for i in range(1, 6)]

# Cross-validation

## Base models

Models that are relatively to cheap to fit using grid search.

In [ ]:
simple_candidates = {
    "Huber": (
        HuberRegressor(max_iter=10000),
        {"epsilon": np.linspace(1, 3, 51).tolist(), "alpha": np.logspace(-4, 0, 101).tolist()}  # we can optimize alpha, in theory, like we do for enet, but that's likely don't worth it
    ),
    "TheilSen": (
        TheilSenRegressor(max_iter=10000),
        {"max_subpopulation": [10000, 50000, 100000], "n_subsamples": [None, 50, 100, 150, 200]}
    ),
}

In [ ]:
best_hp = defaultdict(dict)

In [ ]:
for igp in tqdm_notebook(range(21)):
    est, grid = simple_candidates["Huber"]
    search = GridSearchCV(
        est,
        grid,
        cv=splits,
        scoring={"rsq": rsq, "rmse": rmse, "objective": objective},
        refit="objective",
        n_jobs=-1,
    )
    search.fit(xs_cv, ys_cv[:, igp])
    best = search.best_index_
    best_hp[igp + 1]["Huber"] = (
        search.best_params_,
        search.best_score_,
        {
            "mean_rsq": search.cv_results_["mean_test_rsq"][best],
            "mean_rmse": -search.cv_results_["mean_test_rmse"][best],
            "mean_objective": -search.cv_results_["mean_test_objective"][best],
        },
    )


for igp in range(21):
    est, grid = simple_candidates["TheilSen"]
    search = GridSearchCV(
        est,
        grid,
        cv=splits,
        scoring={"rsq": rsq, "rmse": rmse, "objective": objective},
        refit="objective",
        n_jobs=-1,
    )
    search.fit(xs_cv, ys_cv[:, igp])
    best = search.best_index_
    best_hp[igp + 1]["TheilSen"] = (
        search.best_params_,
        search.best_score_,
        {
            "mean_rsq": search.cv_results_["mean_test_rsq"][best],
            "mean_rmse": -search.cv_results_["mean_test_rmse"][best],
            "mean_objective": -search.cv_results_["mean_test_objective"][best],
        },
    )

## Plain Linear and optimized Enet

OLS does not require CV (only hyperparameter is whether to include intercept, or not, we would like to include it).

Enet require more complicated tuning for alpha parameter, in order enet/regularized regression to degenerate to just OLS (but worse).

In [ ]:
alpha = np.logspace(-4, 1, 2000).tolist()
linear = {"alpha": alpha, "l1_ratio": np.linspace(0, 1, 21).tolist()}

In [ ]:
cv_res_enet = []

for igp in tqdm_notebook(range(21)):
    est, grid = ElasticNet(max_iter=10000), linear
    search = GridSearchCV(
        est,
        grid,
        cv=splits,
        scoring={"rsq": rsq, "rmse": rmse, "objective": objective, "mse": mse},
        refit="objective",
        n_jobs=-1,
    )
    search.fit(xs_cv, ys_cv[:, igp])
    cv_res_enet.append(pd.DataFrame(search.cv_results_).assign(target=f"igp{igp+1}")[[
        "param_alpha",
        "param_l1_ratio",
        "params",
        "mean_test_rsq",
        "std_test_rsq",
        "rank_test_rsq",
        "mean_test_rmse",
        "std_test_rmse",
        "rank_test_rmse",
        "mean_test_objective",
        "std_test_objective",
        "rank_test_objective",
        "mean_test_mse",
        "std_test_mse",
        "rank_test_mse",
        "target",
    ]].dropna(axis=0, how="any"))

In [ ]:
for igp in range(21):
    ratios = {}
    for group, df in cv_res_enet[igp].groupby("param_l1_ratio"):
        argmin = df["mean_test_mse"].argmax()
        alpha_min = df["param_alpha"].iloc[argmin]
        mse_min = df["mean_test_mse"].iloc[argmin]
        se_min = df["std_test_mse"].iloc[argmin] / np.sqrt(5)
        threshold = mse_min - se_min
        one_se_row = df.sort_values("param_alpha").loc[(df.sort_values("param_alpha")["mean_test_mse"] > threshold) & (df.sort_values("param_alpha")["param_alpha"] > alpha_min)].iloc[0]
        ratios[group] = {"mse_min": mse_min, "alpha_min": alpha_min, "mse_1se": one_se_row.mean_test_mse, "alpha_1se": one_se_row.param_alpha}
    all_ratios = pd.DataFrame(ratios)
    best_par = all_ratios.iloc[:, all_ratios.loc["mse_1se"].argmax()]
    best_par = {"alpha": best_par.alpha_1se, "l1_ratio": best_par.name}
    best_scores = cv_res_enet[igp][(cv_res_enet[igp].param_alpha == best_par["alpha"]) & (cv_res_enet[igp].param_l1_ratio == best_par["l1_ratio"])]
    best_hp[igp + 1]["Linear"] = (
        best_par,
        best_scores["mean_test_objective"].iloc[0],
        {
            "mean_rsq": best_scores["mean_test_rsq"].iloc[0],
            "mean_rmse": -best_scores["mean_test_rmse"].iloc[0],
            "mean_objective": -best_scores["mean_test_objective"].iloc[0],
        },
    )

## Random search subset

We use random search for random forrest. Grid search is not feasible, because our grid has total of ~250k combinations. This will take too long. For Random Forest, we usually don't need a huge combinatorial search because increasing `n_estimators` mostly improves stability until diminishing returns, and `max_features`, `max_depth`, and node-splitting parameters are the most influential --- but their optimum points are fairly broad. So we stick to a random search across our grid.

In [ ]:
random = (
    RandomForestRegressor(n_jobs=-1),
    {
        "n_estimators": randint(100, 600),
        "max_depth": [None, 1, 2, 3, 4, 5, 10, 15, 20, 25, 30],
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", 0.5, 0.7, None],
    },
)

In [ ]:
for igp in tqdm_notebook(range(21)):
    est, dist = random
    search = RandomizedSearchCV(
        est,
        dist,
        n_iter=1000,
        cv=splits,
        scoring={"rsq": rsq, "rmse": rmse, "objective": objective},
        refit="objective",
        n_jobs=-1,
    )
    search.fit(xs_cv, ys_cv[:, igp])
    best = search.best_index_
    best_hp[igp + 1]["RF"] = (
        search.best_params_,
        search.best_score_,
        {
            "mean_rsq": search.cv_results_["mean_test_rsq"][best],
            "mean_rmse": -search.cv_results_["mean_test_rmse"][best],
            "mean_objective": -search.cv_results_["mean_test_objective"][best],
        },
    )

## More complex models with more complex CV

Namely, TabNet and boosting.

Since boosting iteratively fits trees (regressions). Additionally, hyperparameters (`learning_rate`, `max_depth`, `subsample`) interact nonlinearly. All of this results in more sensetive tuning, where wrong combos can sink performance significantly. Because of this we will use `optunas`'s TPESampler for optimization.

TaBnet is just too complex for hypermarameters grid to be tractable.

In [ ]:
def xgb_hp(trial):
    return {
        "booster": trial.suggest_categorical("booster", ["gbtree", "dart", "gblinear"]),
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 1337,
        "n_jobs": -1,
    }


def xgb_linreg(trial):
    return {
        "booster": "gblinear",
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "random_state": 1337,
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 1.5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 1.5, log=True),
        "n_jobs": -1,
    }


def xgb_tree_hp(trial):
    return {
        "booster": trial.suggest_categorical("booster", ["gbtree", "dart"]),
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 1337,
        "tree_method": "hist",
        "n_jobs": -1,
    }


def tabnet_hp(trial):
    params = {
        "n_d": trial.suggest_int("n_d", 8, 64),
        "n_a": trial.suggest_int("n_a", 8, 64),
        "n_steps": trial.suggest_int("n_steps", 3, 10),
        "gamma": trial.suggest_float("gamma", 1.0, 2.0),
        "n_independent": trial.suggest_int("n_independent", 1, 5),
        "n_shared": trial.suggest_int("n_shared", 1, 5),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-3, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "mask_type": trial.suggest_categorical("mask_type", ["sparsemax", "entmax"]),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "virtual_batch_size": trial.suggest_categorical("virtual_batch_size", [128, 256]),
        "max_epochs": 200,
        "patience": 20,
        "scheduler_patience": 10,
    }
    params["virtual_batch_size"] = min(params["virtual_batch_size"], params["batch_size"])
    return params

In [ ]:
def cv_objective_xgb(mod, xs, y, splits):
    obj, rmse, rsq = [], [], []
    for train_idx, val_idx in splits:
        xs_train, xs_val = xs[train_idx], xs[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        mod.fit(xs_train, y_train)
        pred = mod.predict(xs_val)
        obj.append(_objective(y_val, pred))
        rsq.append(_rsq(y_val, pred))
        rmse.append(root_mean_squared_error(y_val, pred))
    return np.mean(obj), np.mean(rmse), np.mean(rsq)


def cv_objective_tabnet(mod, params, xs, y, splits):
    obj, rmse, rsq = [], [], []
    for train_idx, val_idx in splits:
        xs_train, xs_val = xs[train_idx], xs[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        mod.fit(
            xs_train,
            y_train.reshape(-1, 1),
            eval_set=[(xs_val, y_val.reshape(-1, 1))],
            max_epochs=params.get("max_epochs"),
            patience=params.get("patience"),
            batch_size=params.get("batch_size"),
            virtual_batch_size=params.get("virtual_batch_size"),
            eval_metric=["rmse"],
        )
        pred = mod.predict(xs_val).flatten()
        obj.append(_objective(y_val, pred))
        rsq.append(_rsq(y_val, pred))
        rmse.append(root_mean_squared_error(y_val, pred))
    return np.mean(obj), np.mean(rmse), np.mean(rsq)

In [ ]:
def xgb_objective(trial, xs, y, splits):
    params = xgb_hp(trial)
    mod = xgb.XGBRegressor(objective="reg:squarederror", **params)
    obj, mean_rsq, mean_rmse = cv_objective_xgb(mod, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj


def xgb_linreg_objective(trial, xs, y, splits):
    params = xgb_linreg(trial)
    mod = xgb.XGBRegressor(objective="reg:squarederror", **params)
    obj, mean_rsq, mean_rmse = cv_objective_xgb(mod, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj


def xgb_tree_objective(trial, xs, y, splits):
    params = xgb_tree_hp(trial)
    mod = xgb.XGBRegressor(objective="reg:squarederror", **params)
    obj, mean_rsq, mean_rmse = cv_objective_xgb(mod, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj


def tabnet_objective(trial, xs, y, splits):
    params = xgb_hp(trial)
    mod = TabNetRegressor(
        n_d=params.get("n_d"),
        n_a=params.get("n_a"),
        n_steps=params.get("n_steps"),
        n_independent=params.get("n_independent"),
        n_shared=params.get("n_shared"),
        gamma=params.get("gamma"),
        lambda_sparse=params.get("lambda_sparse"),
        optimizer_params={"lr": params.get("learning_rate")},
        mask_type=params.get("mask_type"),
        scheduler_params={
            "mode": "min",
            "patience": params.get("scheduler_patience"),
            "min_lr": 1e-5,
            "factor": 0.5,
        },
        seed=1337,
        verbose=0,
    )
    obj, mean_rsq, mean_rmse = cv_objective_tabnet(mod, params, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj

In [ ]:
for igp in tqdm_notebook(range(21)):
    study = optuna.create_study(
        study_name=f"igp{igp+1}",
        storage="sqlite:///xgb/optuna_study_xgb_iGP.db",
        load_if_exists=True,
        direction="minimize",
        sampler=TPESampler(seed=1337+igp, multivariate=True, n_startup_trials=20),
    )
    study.optimize(
        lambda trial: xgb_objective(trial, xs_cv, ys_cv[:,igp], splits),
        n_trials=300,
        callbacks=[
            optuna.study.MaxTrialsCallback(
                310, states=(optuna.trial.TrialState.COMPLETE,)
            )
        ],
    )
    best_hp[igp + 1]["Boost"] = (
        study.best_trial.params,
        study.best_value,
        {
            "mean_rsq": study.best_trial.user_attrs.get("cv_rsq"),
            "mean_rmse": study.best_trial.user_attrs.get("cv_rmse"),
            "mean_objective": study.best_value,
        },
    )

In [ ]:
for igp in tqdm_notebook(range(21)):
    study = optuna.create_study(
        study_name=f"igp{igp+1}",
        storage="sqlite:///xgb/optuna_study_xgb_tree_iGP.db",
        load_if_exists=True,
        direction="minimize",
        sampler=TPESampler(seed=1337+igp, multivariate=True, n_startup_trials=20),
    )
    study.optimize(
        lambda trial: xgb_tree_objective(trial, xs_cv, ys_cv[:,igp], splits),
        n_trials=150,
        callbacks=[
            optuna.study.MaxTrialsCallback(
                150, states=(optuna.trial.TrialState.COMPLETE,)
            )
        ],
    )
    best_hp[igp + 1]["Boost_tree"] = (
        study.best_trial.params,
        study.best_value,
        {
            "mean_rsq": study.best_trial.user_attrs.get("cv_rsq"),
            "mean_rmse": study.best_trial.user_attrs.get("cv_rmse"),
            "mean_objective": study.best_value,
        },
    )

In [ ]:
for igp in tqdm_notebook(range(21)):
    study = optuna.create_study(
        study_name=f"igp{igp+1}",
        storage=f"sqlite:///tabnet/optuna_study_tabnet_iGP{igp+1}.db",
        load_if_exists=True,
        direction="minimize",
        sampler=TPESampler(seed=1337+igp, n_startup_trials=20),
    )
    study.optimize(
        lambda trial: tabnet_objective(trial, xs_cv, ys_cv[:,igp], splits),
        n_trials=300,
        callbacks=[
            optuna.study.MaxTrialsCallback(
                305, states=(optuna.trial.TrialState.COMPLETE,)
            )
        ],
    )
    best_hp[igp + 1]["TabNet"] = (
        study.best_trial.params,
        study.best_value,
        {
            "mean_rsq": study.best_trial.user_attrs.get("cv_rsq"),
            "mean_rmse": study.best_trial.user_attrs.get("cv_rmse"),
            "mean_objective": study.best_value,
        },
    )

In [13]:
os.getcwd()

'd:\\Projects\\GPD\\tabnet'

In [22]:
metrics = defaultdict(list)
for igp, best_mods in best_hp.items():
    huber = HuberRegressor(
        max_iter=10000,
        alpha=best_mods["Huber"][0]["alpha"],
        epsilon=best_mods["Huber"][0]["epsilon"],
    ).fit(xs_cv, ys_cv[:,igp - 1])
    preds = huber.predict(xs_test)
    metrics[igp].append({
        "model": "Huber",
        "rmse": root_mean_squared_error(ys_test[:, igp - 1], preds),
        "rsq": _rsq(ys_test[:, igp - 1], preds),
        "objective": _objective(ys_test[:, igp - 1], preds)
    })
    theil = TheilSenRegressor(
        max_iter=10000,
        max_subpopulation=best_mods["TheilSen"][0]["max_subpopulation"],
        n_subsamples=best_mods["TheilSen"][0]["n_subsamples"],
    ).fit(xs_cv, ys_cv[:,igp - 1])
    preds = theil.predict(xs_test)
    metrics[igp].append({
        "model": "TheilSen",
        "rmse": root_mean_squared_error(ys_test[:, igp - 1], preds),
        "rsq": _rsq(ys_test[:, igp - 1], preds),
        "objective": _objective(ys_test[:, igp - 1], preds)
    })
    rf = RandomForestRegressor(
        n_jobs=-1,
        max_depth=best_mods["RF"][0]["max_depth"],
        max_features=best_mods["RF"][0]["max_features"],
        min_samples_leaf=best_mods["RF"][0]["min_samples_leaf"],
        min_samples_split=best_mods["RF"][0]["min_samples_split"],
        n_estimators=best_mods["RF"][0]["n_estimators"],
    ).fit(xs_cv, ys_cv[:,igp - 1])
    preds = rf.predict(xs_test)
    metrics[igp].append({
        "model": "RF",
        "rmse": root_mean_squared_error(ys_test[:, igp - 1], preds),
        "rsq": _rsq(ys_test[:, igp - 1], preds),
        "objective": _objective(ys_test[:, igp - 1], preds)
    })
    lin = ElasticNet(
        max_iter=10000,
        alpha=best_mods["Linear"][0]["alpha"],
        l1_ratio=best_mods["Linear"][0]["l1_ratio"],
    ).fit(xs_cv, ys_cv[:,igp - 1])
    preds = lin.predict(xs_test)
    metrics[igp].append({
        "model": "Linear",
        "rmse": root_mean_squared_error(ys_test[:, igp - 1], preds),
        "rsq": _rsq(ys_test[:, igp - 1], preds),
        "objective": _objective(ys_test[:, igp - 1], preds)
    })
    lin0 = LinearRegression().fit(xs_cv, ys_cv[:,igp - 1])
    preds = lin0.predict(xs_test)
    metrics[igp].append({
        "model": "OLS",
        "rmse": root_mean_squared_error(ys_test[:, igp - 1], preds),
        "rsq": _rsq(ys_test[:, igp - 1], preds),
        "objective": _objective(ys_test[:, igp - 1], preds)
    })

c:\Users\Denis\miniforge3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.867e+02, tolerance: 1.651e-01
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
c:\Users\Denis\miniforge3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.127e+02, tolerance: 1.698e-01
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeC

In [32]:
pd.DataFrame(metrics[5])

,model,rmse,rsq,objective
0,Huber,0.532460,0.704493,-0.651247
1,TheilSen,0.532348,0.704643,-0.651408
2,RF,0.607748,0.615237,-0.554462
3,Linear,0.534848,0.701196,-0.647711
4,OLS,0.532193,0.704422,-0.651203


In [19]:
best_hp

defaultdict(dict,
            {1: {'Huber': ({'alpha': 0.001584893192461114, 'epsilon': 1.0},
               np.float64(0.48788253835992224),
               {'mean_rsq': np.float64(0.5541687044799879),
                'mean_rmse': np.float64(0.6628616612006559),
                'mean_objective': np.float64(-0.48788253835992224)}),
              'TheilSen': ({'max_subpopulation': 100000, 'n_subsamples': 200},
               np.float64(0.48231488910148795),
               {'mean_rsq': np.float64(0.5490212292879777),
                'mean_rmse': np.float64(0.667063401864897),
                'mean_objective': np.float64(-0.48231488910148795)}),
              'RF': ({'max_depth': None,
                'max_features': 0.7,
                'min_samples_leaf': 3,
                'min_samples_split': 4,
                'n_estimators': 487},
               np.float64(0.4192386300974634),
               {'mean_rsq': np.float64(0.4905890042476999),
                'mean_rmse': np.float64(0.713503

In [ ]:
metrics = defaultdict(list)
for igp, df in stats.groupby("iGP"):
    for _, row in df.iterrows():
        if row["method"] == "Huber":
            mod = HuberRegressor(max_iter=10000, **eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if row["method"] == "TheilSen":
            mod = TheilSenRegressor(max_iter=10000,**eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if row["method"] == "Linear":
            mod = ElasticNet(max_iter=10000, **eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if row["method"] == "RF":
            mod = RandomForestRegressor(n_jobs=-1, **eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if row["method"] in ["Boost_reg", "Boost_tree"]:
            mod = xgb.XGBRegressor(objective="reg:squarederror", **eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if row["method"] in ["tabnet"]:
            mod = obj[igp-1]["model"]
        preds = mod.predict(xs_test).flatten()
        rmse = root_mean_squared_error(ys_test[:, igp - 1], preds)
        rsq = _rsq(ys_test[:, igp - 1], preds)
        ci = bootstrap((ys_test[:, igp - 1], preds), _rsq, confidence_level=0.95, paired=True)
        metrics[igp].append({
        "method": row["method"],
        "rmse": rmse,
        "rsq": rsq,
        "rsq_low": ci.confidence_interval.low,
        "rsq_high": ci.confidence_interval.high
        })
    mod = LinearRegression().fit(xs_cv, ys_cv[:,igp - 1])
    preds = mod.predict(xs_test)
    rmse = root_mean_squared_error(ys_test[:, igp - 1], preds)
    rsq = _rsq(ys_test[:, igp - 1], preds)
    ci = bootstrap((ys_test[:, igp - 1], preds), _rsq, confidence_level=0.95, paired=True)
    metrics[igp].append({
        "method": "OLS",
        "rmse": rmse,
        "rsq": rsq,
        "rsq_low": ci.confidence_interval.low,
        "rsq_high": ci.confidence_interval.high
    })

In [ ]:
## {1: }

coefs = defaultdict(list)
for meth, df in cvs.groupby("method"):
    for _, row in df.iterrows():
        igp = row["iGP"]
        if meth == "Huber":
            mod = HuberRegressor(max_iter=10000, **eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if meth == "TheilSen":
            mod = TheilSenRegressor(max_iter=10000,**eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        if meth == "Linear":
            mod = ElasticNet(max_iter=10000, **eval(row["hp"])).fit(xs_cv, ys_cv[:,igp - 1])
        coefs[meth].append(np.array([mod.intercept_, *mod.coef_]))

for igp in range(1,22):
    mod = LinearRegression().fit(xs_cv, ys_cv[:,igp - 1])
    coefs["OLS"].append(np.array([mod.intercept_, *mod.coef_]))